# MoodMate: Effect Size and Power Analysis

A hypothesis-driven 30-day study of the claim that daily emotion logging reduces the chance of depressive symptoms by 25%. Prepared for AIPI 510, Week 6.

This notebook is a prospective design exercise. All event rates below are planning assumptions, not observed MoodMate data.

## Research question and design

Among adults eligible to use MoodMate, does daily emotion logging for 30 days reduce the proportion who screen positive for depressive symptoms at day 30, compared with a matched neutral check-in?

Use an individually randomized, parallel-group trial with 1:1 allocation. The intervention group logs emotions daily. The active-control group completes a neutral wellbeing check-in on the same schedule, with the same reminders and app exposure. Matching contact helps separate the logging feature from attention and notification effects. Randomize after baseline assessment and stratify on baseline PHQ-9 screening status.

Measure symptoms with the PHQ-9 at baseline and day 30. Define the binary primary outcome as PHQ-9 >= 10 at day 30, a positive symptom screen rather than a clinical diagnosis. The threshold is commonly used to indicate clinically significant depressive symptoms (Kroenke et al., 2001). Include adults with a range of baseline symptoms; the day-30 comparison is prevalence, not new-onset incidence. Use intention-to-treat assignment, document missing follow-up, and prespecify missing-data sensitivity analyses. Provide clear crisis/referral guidance and monitor symptom worsening in both arms.

## Hypotheses and statistical test

Let p_M be the day-30 positive-screen risk in MoodMate and p_C the risk in the active control.

- Null: H0: p_M >= p_C (MoodMate does not lower the risk).
- Alternative: H1: p_M < p_C (MoodMate lowers the risk).

Use a prespecified one-sided two-independent-proportions score test at alpha = 0.05. A one-sided test matches the directional client claim; it must be selected before seeing outcomes. Separately report the risk difference and risk ratio with confidence intervals, and monitor possible worsening/adverse outcomes rather than treating the efficacy test as a safety test. Equal randomization supports the planned comparison. A baseline-adjusted logistic model can be a supportive analysis, but the transparent primary power calculation below uses unadjusted proportions.

## Literature and effect-size choice

Firth et al. (2017) synthesized 18 randomized trials of 22 smartphone apps, with 3,414 participants. The pooled effect on depressive symptom severity was Hedges' g = 0.38 (95% CI 0.24 to 0.52). In trials with active controls, the pooled effect was smaller: g = 0.22 (95% CI 0.10 to 0.33). The review also reported that the presence of mood-monitoring software did not significantly moderate study effects.

This is indirect evidence: the review pooled continuous symptom scales across diverse apps and interventions, not a logging-only app's binary PHQ-9 outcome at exactly 30 days. Therefore, use the client's 25% relative-risk claim as the primary minimum effect of interest. Use the active-control g = 0.22 only for a clearly labeled rough binary-outcome sensitivity translation below, not as a precise forecast for MoodMate.

Reference: Firth J, Torous J, Nicholas J, et al. The efficacy of smartphone-based mental health interventions for depressive symptoms: a meta-analysis of randomized controlled trials. *World Psychiatry*. 2017;16(3):287-298. https://doi.org/10.1002/wps.20472

PHQ-9 reference: Kroenke K, Spitzer RL, Williams JBW. The PHQ-9: validity of a brief depression severity measure. *Journal of General Internal Medicine*. 2001;16(9):606-613. https://doi.org/10.1046/j.1525-1497.2001.016009606.x

In [1]:
from math import asin, ceil, exp, pi, sqrt
from statistics import NormalDist

normal = NormalDist()

def cohens_h(p_treatment, p_control):
    """Cohen's h; negative values indicate a lower treatment event rate."""
    return 2 * asin(sqrt(p_treatment)) - 2 * asin(sqrt(p_control))

def required_n_per_arm(p_control, p_treatment, alpha=0.05, target_power=0.80):
    """Normal-approximation sample size for a one-sided, equal-allocation test."""
    pooled = (p_control + p_treatment) / 2
    z_alpha = normal.inv_cdf(1 - alpha)
    z_power = normal.inv_cdf(target_power)
    numerator = (
        z_alpha * sqrt(2 * pooled * (1 - pooled))
        + z_power * sqrt(
            p_control * (1 - p_control)
            + p_treatment * (1 - p_treatment)
        )
    ) ** 2
    return ceil(numerator / (p_control - p_treatment) ** 2)

def approximate_power(n_per_arm, p_control, p_treatment, alpha=0.05):
    """Approximate one-sided score-test power at equal group sizes."""
    pooled = (p_control + p_treatment) / 2
    se_null = sqrt(2 * pooled * (1 - pooled) / n_per_arm)
    se_alternative = sqrt(
        (p_control * (1 - p_control) + p_treatment * (1 - p_treatment))
        / n_per_arm
    )
    z_alpha = normal.inv_cdf(1 - alpha)
    z_effect = (p_control - p_treatment) / se_alternative
    return normal.cdf(z_effect - z_alpha * se_null / se_alternative)

## Primary power analysis: the client's claim

Assume that 20% of active-control participants will have a positive PHQ-9 screen at day 30. This is an explicit planning assumption, not a prevalence estimate established by the cited meta-analysis. Replace it with pilot or population-specific data before launching a real trial. A 25% relative reduction means p_M = 0.75 x 0.20 = 0.15, an absolute reduction of 5 percentage points.

The primary effect size is Cohen's h computed from those two proportions. The calculation uses equal allocation, one-sided alpha = 0.05, 80% power, and a normal approximation to a two-proportion score test. It finds the analyzable sample size per arm; 15% is then added for expected loss to follow-up.

In [2]:
alpha = 0.05
target_power = 0.80
attrition = 0.15
control_risk = 0.20
claimed_relative_risk = 0.75
moodmate_risk = control_risk * claimed_relative_risk

primary_n_per_arm = required_n_per_arm(
    control_risk, moodmate_risk, alpha=alpha, target_power=target_power
)
recruited_n_per_arm = ceil(primary_n_per_arm / (1 - attrition))
primary_h = cohens_h(moodmate_risk, control_risk)
achieved_power = approximate_power(
    primary_n_per_arm, control_risk, moodmate_risk, alpha=alpha
)

print(f"Control day-30 positive-screen risk: {control_risk:.1%}")
print(f"MoodMate risk under the claim: {moodmate_risk:.1%}")
print(f"Absolute risk reduction: {control_risk - moodmate_risk:.1%}")
print(f"Relative risk: {moodmate_risk / control_risk:.2f}")
print(f"Cohen's h: {primary_h:.3f}")
print(f"Analyzable sample required: {primary_n_per_arm} per arm ({2 * primary_n_per_arm} total)")
print(f"Recruit after {attrition:.0%} attrition inflation: {recruited_n_per_arm} per arm ({2 * recruited_n_per_arm} total)")
print(f"Approximate achieved power at the rounded sample size: {achieved_power:.3f}")

Control day-30 positive-screen risk: 20.0%
MoodMate risk under the claim: 15.0%
Absolute risk reduction: 5.0%
Relative risk: 0.75
Cohen's h: -0.132
Analyzable sample required: 714 per arm (1428 total)
Recruit after 15% attrition inflation: 840 per arm (1680 total)
Approximate achieved power at the rounded sample size: 0.800


## Literature-based sensitivity translation

For a rough comparison only, translate the active-control Hedges' g = 0.22 into an odds ratio using log(OR) approximately equal to -g x pi / sqrt(3), an approximation that assumes logistic distributions for the continuous and binary outcomes. Convert that odds ratio to a day-30 risk using the same 20% control-risk assumption. This conversion is not a substitute for evidence on MoodMate's binary endpoint; it illustrates how the result depends on strong distributional assumptions. The primary design remains powered for the smaller, explicit client claim (RR = 0.75).

In [3]:
active_control_g = 0.22
literature_or = exp(-active_control_g * pi / sqrt(3))
literature_implied_risk = (
    literature_or * control_risk
    / (1 - control_risk + literature_or * control_risk)
)
literature_rr = literature_implied_risk / control_risk
literature_n_per_arm = required_n_per_arm(
    control_risk, literature_implied_risk, alpha=alpha, target_power=target_power
)

print(f"Approximate odds ratio from g={active_control_g:.2f}: {literature_or:.3f}")
print(f"Implied treatment risk at control risk {control_risk:.1%}: {literature_implied_risk:.1%}")
print(f"Implied relative risk: {literature_rr:.3f}")
print(f"Cohen's h for this rough translation: {cohens_h(literature_implied_risk, control_risk):.3f}")
print(f"Approximate analyzable n per arm for this translated effect: {literature_n_per_arm}")
print("Interpret this as an indirect sensitivity calculation, not as a verified binary effect for MoodMate.")

Approximate odds ratio from g=0.22: 0.671
Implied treatment risk at control risk 20.0%: 14.4%
Implied relative risk: 0.718
Cohen's h for this rough translation: -0.150
Approximate analyzable n per arm for this translated effect: 554
Interpret this as an indirect sensitivity calculation, not as a verified binary effect for MoodMate.


## Sensitivity to the control event rate

Required sample size depends on the day-30 control risk even when the claimed relative risk stays at 0.75. The table below varies that baseline risk from 10% to 30%. Each total includes 15% attrition inflation. This is why pilot data for the intended population matter.

In [4]:
control_risks = [0.10, 0.15, 0.20, 0.25, 0.30]
print(f"{'Control risk':>13} {'MoodMate risk':>14} {'h':>8} {'n/arm':>8} {'Recruit total':>15}")
for p_control in control_risks:
    p_treatment = claimed_relative_risk * p_control
    n_per_arm = required_n_per_arm(
        p_control, p_treatment, alpha=alpha, target_power=target_power
    )
    total_recruited = 2 * ceil(n_per_arm / (1 - attrition))
    print(
        f"{p_control:12.0%} {p_treatment:13.1%} "
        f"{cohens_h(p_treatment, p_control):8.3f} {n_per_arm:8d} {total_recruited:15d}"
    )

 Control risk  MoodMate risk        h    n/arm   Recruit total
         10%          7.5%   -0.089     1579            3716
         15%         11.2%   -0.111     1002            2358
         20%         15.0%   -0.132      714            1680
         25%         18.8%   -0.152      540            1272
         30%         22.5%   -0.171      425            1000


## Interpretation and rationale

Under the stated assumptions, the study needs the computed analyzable number in each arm and should recruit the attrition-adjusted total shown above. The client claim corresponds to a 5-percentage-point absolute decrease when control risk is 20%. The primary Cohen's h and sample size are derived from that claim, while the published continuous Hedges' g is used only as an indirect benchmark. The control-risk sensitivity table shows that the recruitment target changes substantially with the underlying event rate.

The active neutral-check-in control balances reminders and engagement. The PHQ-9 provides a repeatable symptom measure, but a positive screen is not a diagnosis. A randomized comparison can estimate the effect of offering the logging feature in the enrolled population; it cannot establish effectiveness for all users or clinical diagnosis. Missing outcomes, adherence, baseline symptom mix, and self-report expectations can affect conclusions and should be reported. Before fielding the study, validate the assumed 20% control risk and 15% attrition using a pilot or relevant local data.